# Exercício 4 — Outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

df = sns.load_dataset("diamonds")
df.head()

## Estatísticas e valores impossíveis

In [ ]:
numericas = df.select_dtypes(include=np.number).columns
descricao = df[numericas].describe().T
impossiveis = {
    "carat_menor_igual_zero": int((df["carat"] <= 0).sum()),
    "x_menor_igual_zero": int((df["x"] <= 0).sum()),
    "y_menor_igual_zero": int((df["y"] <= 0).sum()),
    "z_menor_igual_zero": int((df["z"] <= 0).sum()),
    "price_menor_igual_zero": int((df["price"] <= 0).sum())
}
descricao, impossiveis

## Detecção por Z-score

In [ ]:
z_flags = pd.DataFrame(np.abs(zscore(df[numericas], nan_policy="omit")) > 3, columns=numericas, index=df.index)
quantidade_z = z_flags.sum().sort_values(ascending=False)
quantidade_z

## Detecção por IQR

In [ ]:
limites_iqr = {}
iqr_flags = pd.DataFrame(False, index=df.index, columns=numericas)
for coluna in numericas:
    q1, q3 = df[coluna].quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    limites_iqr[coluna] = (inferior, superior)
    iqr_flags[coluna] = (df[coluna] < inferior) | (df[coluna] > superior)
quantidade_iqr = iqr_flags.sum().sort_values(ascending=False)
pd.DataFrame({"z_score": quantidade_z, "iqr": quantidade_iqr})

## Visualizações

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=df[numericas], ax=eixos[0])
eixos[0].tick_params(axis="x", rotation=70)
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x="carat", y="price", alpha=.35, ax=eixos[1])
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x="x", y="y", hue="cut", alpha=.35, ax=eixos[2])
plt.tight_layout()
plt.show()

## Critério de classificação

In [ ]:
erros_fisicos = df[(df["x"] <= 0) | (df["y"] <= 0) | (df["z"] <= 0)]
limites_price = limites_iqr["price"]
valores_extremos_legitimos = df[df["carat"] > df["carat"].quantile(.99)]
print("Registros com dimensões fisicamente impossíveis:", len(erros_fisicos))
print("Registros no percentil 99 de carat:", len(valores_extremos_legitimos))

## Tratamentos

In [ ]:
df_remocao = df.loc[~((df["x"] <= 0) | (df["y"] <= 0) | (df["z"] <= 0))].copy()
df_capping = df.copy()
for coluna, (inferior, superior) in limites_iqr.items():
    df_capping[coluna] = df_capping[coluna].clip(inferior, superior)
df_log = df.copy()
for coluna in numericas:
    df_log[coluna] = np.log1p(df_log[coluna].clip(lower=0))
comparacao = pd.DataFrame({"original": df["price"].describe(), "remocao": df_remocao["price"].describe(), "capping": df_capping["price"].describe(), "log": df_log["price"].describe()})
comparacao

## Distribuição de price após os tratamentos

In [ ]:
fig, eixos = plt.subplots(1, 4, figsize=(16, 4))
for eixo, base, titulo in zip(eixos, [df, df_remocao, df_capping, df_log], ["Original", "Remoção", "Capping", "Log"]):
    sns.histplot(base["price"], bins=40, ax=eixo)
    eixo.set_title(titulo)
plt.tight_layout()
plt.show()

## Respostas de reflexão

O Z-score se baseia em uma referência de média e desvio-padrão e funciona melhor quando a distribuição não é muito assimétrica. Price apresenta cauda à direita, então o gráfico é importante antes de usar um limite fixo de Z-score.

Um diamante com mais de quatro quilates pode ser raro e legítimo. Para um modelo de preço de diamantes, removê-lo pode eliminar justamente parte importante do mercado; para um modelo de comportamento típico, pode ser necessário separar esse segmento ou usar uma transformação.

Remover outliers pode esconder fraude, falhas de fabricação ou clientes de altíssimo valor. Em detecção de fraude, os valores extremos são muitas vezes o alvo da análise e não um ruído a ser apagado.